In [ ]:
"""
Code for Appendix A: Description of statistical testing and uncertainty estimation methods
-------------------------------------------------------------------------------------------

This notebook contains the code of the statistical tests and uncertainty estimates of 'Appendix A'. 
It is the producer of several files in data/processed in the root of this repository, which are read 
by appendix_B.ipynb, appendix_C.ipynb and plotting.ipynb: the composite differences with significance 
(ENSO_SSTA_COMP_DIFF_RESULTS_WITH_SIG.nc, ENSO_N34_HOV_COMP_DIFF_RESULTS_WITH_SIG.nc), the extreme-
temperature bootstrap files (EXT_TEMP_BOOTSTRAP/) and the power spectral analysis results (PSD_PEAK_TEST_RED_NOISE.pkl, 
PSD_PEAK_SHIFT_TEST_PHASE_RANDOM.pkl). Its inputs were created by enso_preprocessing.py. The bootstrap 
and surrogate tests are slow, so the notebook is not meant to be re-run in full; the extreme-temperature 
bootstrap is a copy of a stand-alone SLURM script (see Appendix A.2).

Author: Sreevathsa G. (sg13n23@soton.ac.uk; ORCID ID: 0000-0003-4084-9677)
Last updated: 25 November 2026
"""

In [ ]:
# Standard library
from itertools import product
from pathlib import Path
import pickle
import sys
import warnings

# Third-party
import numpy as np
import pandas as pd
from scipy.signal import detrend
from tqdm.notebook import tqdm
import xarray as xr

# Local modules
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import compute_psd, renamer

# Runtime configuration
warnings.simplefilter(action="ignore", category=FutureWarning)

In [13]:
# Directories and files (relative to this notebook)
DATA_DIR = '../data/processed/'
COMP_DIR = DATA_DIR + 'ENSO_TEMPANOM_COMPOSITES/' # Per-dataset ENSO composites (enso_preprocessing.py) and the event-pairing CSVs
HOV_FILE = DATA_DIR + 'N34_HOV_ZVT_TOP300M_{period_label}.nc' # Niño 3.4 depth-time anomalies; fill in the period label

# Period labels used in file names: HIST_EVAL (1979-2014; CTRL, HIST, NPD and OBS), FUTR_EVAL (2015-2050; SCEN, i.e. the full
# scenario 'SCEN_F') and TRNC_SCEN (2015-2040; SCEN, i.e. the truncated scenario 'SCEN_T')
PERIOD_LABELS = {'HIST': 'HIST_EVAL_1979-2014', 'SCEN_F': 'FUTR_EVAL_2015-2050', 'SCEN_T': 'TRNC_SCEN_2015-2040'}

# Dataset name lists
HIGHRESMIP_MODELS = [
    "HadGEM3-GC31-HH", "EC-Earth3P-HR", "CNRM-CM6-1-HR",
    "CMCC-CM2-VHR4", "MPI-ESM1-2-HR", "MPI-ESM1-2-XR"
]
# Experiment names follow the "<model>_<experiment>" convention (CTRL = control-1950, HIST = hist-1950, SCEN = highres-future)
HIGHRESMIP_EXPS_CTRL = [f"{model}_CTRL" for model in HIGHRESMIP_MODELS]
HIGHRESMIP_EXPS_HIST = [f"{model}_HIST" for model in HIGHRESMIP_MODELS]
HIGHRESMIP_EXPS_SCEN = [f"{model}_SCEN" for model in HIGHRESMIP_MODELS]

NPD_EXPS = ["NPD_eORCA025_ERA5", "NPD_eORCA025_JRA55"] # NEMO ocean-only runs forced with ERA5 / JRA55 atmospheres
OBS = ["EN4", "ORAS5"] # Observation-based ocean (re)analyses

# Settings shared by the tests below
N_BOOT = 2000 # Number of bootstrap draws; also in the bootstrap file names (e.g. _b2000s14.nc), which appendix_B.ipynb and plotting.ipynb glob for
ALPHA = 0.05 # Significance level: a result counts as significant when p <= ALPHA

### Finding common ENSO events: NPD and OBS groups

In [14]:
# =============================================================================
# PAIRING ENSO EVENTS ACROSS THE DATASETS OF THE OBS AND NPD GROUPS
# =============================================================================
# Helper function to extract the metadata of each event-level composite
def meta(ds, tag):
    """
    Extract the metadata of each event-level composite as a DataFrame, with columns suffixed by the dataset tag.
    """
    return pd.DataFrame({'phase': ds.event_type.values, # 'EL-NINO' or 'LA-NINA'
                         f'id_{tag}': ds.event_id.values, # event_id within this dataset (a serial index, not comparable across datasets)
                         f'reg_{tag}': ds.event_reg_type.values, # 'CP', 'EP' or 'Mixed'
                         f'start_{tag}': pd.PeriodIndex(ds.event_start.values, freq='M'), # first month of the ENSO event
                         f'end_{tag}': pd.PeriodIndex(ds.event_end.values, freq='M')}) # last month of the ENSO event

# Helper function to pair the ENSO events of two datasets of the same group
def pair_events(d_list = ['ORAS5', 'EN4']):
    """
    Pair the ENSO events of two datasets of the same group (OBS or NPD) for event-matched group means.

    Every event of one dataset is combined with every event of the same phase in the other dataset, and kept as a pair
    only if their month ranges overlap (latest start <= earliest end). Events found in only one dataset are kept as
    single-dataset rows, so each event appears exactly once.

    Parameters
    ----------
    d_list (list of str) : Tags of the two datasets, e.g. ['EN4', 'ORAS5'] or ['NPD_eORCA025_ERA5', 'NPD_eORCA025_JRA55'].
                           Default is ['ORAS5', 'EN4'].

    Returns
    -------
    pd.DataFrame : One row per consolidated event: 'pair_id' (0..n-1, which becomes the event_id after paired_mean), 'phase',
                   the id, regime type and start / end of the event in each dataset (<NA> id where the event is absent),
                   and the consolidated 'start' / 'end' (overlap of the two windows for pairs, the event's own window otherwise).
    """
    ds = {d: renamer(xr.open_dataset(f'{COMP_DIR}ENSO_TEMPANOM_COMPOSITES_{d}_{PERIOD_LABELS["HIST"]}.nc')) for d in d_list}
    meta_A, meta_B = meta(ds[d_list[0]], tag = d_list[0]), meta(ds[d_list[1]], tag = d_list[1])
    print('\n Considering datasets:', d_list[0], 'and', d_list[1], 'for event pairing...')
    print('='*60)

    # Paired events: all El Niño / La Niña combinations of the two datasets (same phase) ...
    pairs   = meta_A.merge(meta_B, on='phase')
    # ... of which only those whose month ranges overlap are kept: latest start <= earliest end
    pairs['start'] = np.maximum(pairs[f'start_{d_list[0]}'], pairs[f'start_{d_list[1]}'])
    pairs['end']   = np.minimum(pairs[f'end_{d_list[0]}'],   pairs[f'end_{d_list[1]}'])
    overlap = pairs['start'] <= pairs['end']
    pairs   = pairs[overlap].reset_index(drop=True)
    print(f'Number of ENSO events in:\n{d_list[0]}:', len(meta_A), f'\n{d_list[1]}:', len(meta_B), '\nPaired (overlapping) events:', len(pairs))
    pairs.insert(0, 'pair_id', pairs.index)

    # Events found in only one dataset (typically short-lived: on average 5-6 months, against 10-12 months for the paired
    # windows) are kept as extra rows with their own window
    only_in_A = meta_A[~meta_A[f'id_{d_list[0]}'].isin(pairs[f'id_{d_list[0]}'])].copy()
    only_in_B = meta_B[~meta_B[f'id_{d_list[1]}'].isin(pairs[f'id_{d_list[1]}'])].copy()
    only_in_A['start'], only_in_A['end'] = only_in_A[f'start_{d_list[0]}'], only_in_A[f'end_{d_list[0]}']
    only_in_B['start'], only_in_B['end'] = only_in_B[f'start_{d_list[1]}'], only_in_B[f'end_{d_list[1]}']
    events = pd.concat([pairs, only_in_A, only_in_B], ignore_index=True)
    events = events.astype({f'id_{d_list[0]}': 'Int64', f'id_{d_list[1]}': 'Int64', 'pair_id': 'Int64'}) # Nullable integers, as the ids are <NA> for absent events
    events['pair_id'] = events.index

    print('\nFinal number of ENSO events across d_list, after pairing:')
    print('='*60)
    print(f'{len(pairs)} paired + {len(only_in_A)} {d_list[0]}-only + {len(only_in_B)} {d_list[1]}-only = {len(events)} events  '
            f'({len(meta_A)} {d_list[0]}, {len(meta_B)} {d_list[1]}) | per-phase: {events.groupby("phase").size().to_dict()}\n')

    print('Mismatch in ENSO regime classification between the two datasets:')
    print('='*60)
    print(pairs.loc[pairs[f'reg_{d_list[0]}'] != pairs[f'reg_{d_list[1]}'], ['pair_id', 'phase', f'reg_{d_list[0]}', f'reg_{d_list[1]}']]) # CP / EP / Mixed disagreements

    return events

# Pairing tables of the OBS and NPD groups (HIST_EVAL only), saved for paired_mean
event_pairs = {'OBS':pair_events(d_list = OBS), 'NPD':pair_events(d_list = NPD_EXPS)}
event_pairs['OBS'].to_csv(f'{COMP_DIR}OBS_EVENT_PAIRS_{PERIOD_LABELS["HIST"]}.csv', index=False)
event_pairs['NPD'].to_csv(f'{COMP_DIR}NPD_EVENT_PAIRS_{PERIOD_LABELS["HIST"]}.csv', index=False)

# Helper function to average the paired events of the two datasets of a group
def paired_mean(ds = None, phase = 'EL-NINO'):
    """
    Compute the mean of paired events across the two datasets of the same group (OBS or NPD).

    For paired events, the mean is taken across both datasets. For single-dataset events, it is the map of the dataset where the
    event is present (the absent dataset contributes NaN). The pairing table is read from the CSV written above.

    Parameters
    ----------
    ds (xr.DataArray) : Event-level composites of both datasets, stacked along 'dataset'; the group (OBS or NPD) is identified
                        from the dataset names.
    phase (str)       : 'EL-NINO' or 'LA-NINA'. Default is 'EL-NINO'.

    Returns
    -------
    xr.DataArray : Mean composite with dim 'event_id' (= pair_id, one entry per consolidated event of this phase) and an
                   'event_type' coordinate.
    """
    group_name = 'OBS' if set(ds.dataset.values) == set(OBS) else 'NPD'
    pairs = pd.read_csv(f'{COMP_DIR}{group_name}_EVENT_PAIRS_{PERIOD_LABELS["HIST"]}.csv')
    pairs = pairs[pairs.phase == phase].reset_index(drop=True)

    d_list = OBS if group_name == 'OBS' else NPD_EXPS
    members = []
    for d in d_list:
        rows = pairs[pairs[f'id_{d}'].notna()] # events present in this dataset
        ids  = rows[f'id_{d}'].astype(int).values
        # Select this dataset's events, relabel them with pair_id, and place them on the full pair_id axis (NaN where absent)
        members.append(ds.sel(dataset=d, event_id=ids).assign_coords(event_id=rows.pair_id.values)
                    .reindex(event_id=pairs.pair_id.values))
    # Stack over the two datasets; join='exact' raises an error if the grids differ instead of NaN-filling
    both = xr.concat(members, dim='dataset', join='exact')
    is_pair = xr.DataArray((pairs[f'id_{d_list[0]}'].notna() & pairs[f'id_{d_list[1]}'].notna()),
                        dims='event_id', coords={'event_id': pairs.pair_id})
    # Paired events: mean of both datasets; skipna=False keeps a grid point masked if it is masked in either dataset.
    # Single-dataset events: that dataset's map (the absent dataset is all-NaN, so skipna=True returns the other)
    mean = both.mean('dataset', skipna=False).where(is_pair, both.mean('dataset', skipna=True))
    return mean.assign_coords(event_type=('event_id', pairs.phase.to_numpy(dtype=object)))


 Considering datasets: EN4 and ORAS5 for event pairing...
Number of ENSO events in:
EN4: 20 
ORAS5: 19 
Paired (overlapping) events: 18

Final number of ENSO events across d_list, after pairing:
18 paired + 2 EN4-only + 1 ORAS5-only = 21 events  (20 EN4, 19 ORAS5) | per-phase: {'EL-NINO': 10, 'LA-NINA': 11}

Mismatch in ENSO regime classification between the two datasets:
    pair_id    phase reg_EN4 reg_ORAS5
2         2  EL-NINO      CP        EP
3         3  EL-NINO      CP     Mixed
10       10  LA-NINA   Mixed        CP
12       12  LA-NINA   Mixed        CP

 Considering datasets: NPD_eORCA025_ERA5 and NPD_eORCA025_JRA55 for event pairing...
Number of ENSO events in:
NPD_eORCA025_ERA5: 20 
NPD_eORCA025_JRA55: 16 
Paired (overlapping) events: 14

Final number of ENSO events across d_list, after pairing:
14 paired + 6 NPD_eORCA025_ERA5-only + 2 NPD_eORCA025_JRA55-only = 22 events  (20 NPD_eORCA025_ERA5, 16 NPD_eORCA025_JRA55) | per-phase: {'EL-NINO': 11, 'LA-NINA': 11}

Mismatch i

## Appendix A.1: Group-mean composites and differences: ENSO SST anomaly and Niño 3.4 depth-time Hovmöller diagrams

Two-level bootstrap for composite-derived quantities: SSTA composite difference maps, Hovmöllers and extreme-anomaly profiles

In [ ]:
# =============================================================================
# LOADING AND EXTREME-ANOMALY HELPERS FOR THE HIERARCHICAL BOOTSTRAP
# =============================================================================
# Helper function to load the event-level composites of one dataset and ENSO phase
def load_data(dataset, phase, kind='sst', lat=slice(-30, 30), lon=slice(120, -70 + 360),
              depth=slice(0, 316), interp_depth=None, hov_lags=np.arange(-14, 9), period_label = None):
    """
    Load the event-level ENSO composites of one dataset and phase as a DataArray with a leading 'event_id' dimension.

    The 'event_type' and 'event_peak' coordinates are attached, ready for hierarchical_bootstrap.

    Parameters
    ----------
    dataset (str)             : Dataset name, e.g. 'HadGEM3-GC31-HH_CTRL', 'EC-Earth3P-HR_SCEN' or 'ORAS5'.
    phase (str)               : 'EL-NINO' or 'LA-NINA'.
    kind (str)                : 'sst' (surface SSTA composites, dims (event_id, lat, lon)), '3d' (full-column composites,
                                dims (event_id, depth, lat, lon)) or 'hov' (event-centred Niño 3.4 Hovmöllers, dims
                                (event_id, time, depth), with time as the lag in months). Default is 'sst'.
    lat, lon (slice)          : Geographical crop for 'sst' and '3d'. Default is the tropical Pacific, 30S-30N and 120E-70W.
    depth (slice)             : Depth range (m) kept for '3d' and 'hov'. Default is 0-316 m.
    interp_depth (np.ndarray) : Target depths (m) for '3d' and 'hov': linear interpolation, then a backward fill along depth
                                so that 0 m takes the shallowest native value. Default is None (no interpolation).
    hov_lags (np.ndarray)     : Months relative to the event peak kept for 'hov'. Default is -14 to +8.
    period_label (str)        : Period of the composite file, one of the PERIOD_LABELS values ('HIST_EVAL_1979-2014',
                                'FUTR_EVAL_2015-2050' or 'TRNC_SCEN_2015-2040').

    Returns
    -------
    xr.DataArray : Composites of the events of this phase, with 'event_type' and 'event_peak' coordinates along 'event_id'.
    """
    # Composite file: data for 'sst' and '3d'; event numbering and timing for all kinds
    comp = renamer(xr.open_dataset(f'{COMP_DIR}ENSO_TEMPANOM_COMPOSITES_{dataset}_{period_label}.nc'))
    comp = comp.squeeze('dataset', drop=True) # singleton dataset axis is not a sampling dim
    comp = comp.where(comp['event_type'] == phase, drop=True)

    if kind == 'sst':
        da = comp['event_level_composite'].sel(lat=lat, lon=lon).isel(depth=0, drop = True)

    elif kind == '3d':
        da = comp['event_level_composite'].sel(lat=lat, lon=lon, depth=depth)

    elif kind == 'hov':
        # Full Niño 3.4 depth-time record of this dataset; cut a window of hov_lags months around each peak
        hov_src = xr.open_dataset(HOV_FILE.format(period_label=period_label))['anomaly'].sel(dataset=dataset, depth=depth)
        peaks   = pd.DatetimeIndex(comp['event_peak'].values)
        hov_list = []
        for pk, eid in zip(peaks, comp['event_id'].values):
            window = pd.date_range(pk + pd.DateOffset(months=int(hov_lags[0])), periods=len(hov_lags), freq='MS')
            hov_list.append(hov_src.reindex(time=window) # months beyond the record come back NaN
                                   .assign_coords(time=hov_lags) # absolute time -> lag (months)
                                   .expand_dims(event_id=[eid]))
        da = xr.concat(hov_list, dim='event_id').transpose('event_id', 'time', 'depth')

    else:
        raise ValueError("kind must be 'sst', '3d' or 'hov'")

    # Only the OBS products are interpolated here: EN4 and ORAS5 are on different depth grids and need a common one for
    # paired_mean() (called from load_group() in hierarchical_bootstrap). The NPD runs share a depth grid and are not interpolated.
    # For '3d', the other datasets are interpolated in extreme_anomaly_operator(), after the extreme anomalies are masked.
    # For 'hov', the source file is already on the 0.5 m grid of all datasets
    INTERP_DATASETS = {'EN4', 'ORAS5'}

    if kind in ('3d', 'hov') and interp_depth is not None and dataset in INTERP_DATASETS:
        da = da.interp(depth=interp_depth).bfill(dim='depth')

    # Carry the event metadata on every kind ('hov' loses it when cut from hov_src)
    return da.assign_coords(event_type=('event_id', comp['event_type'].values),
                            event_peak=('event_id', comp['event_peak'].values))

# Helper function to compute the extent and intensity profiles of extreme anomalies in the Niño 3.4 region
def extreme_anomaly_operator(ds, phase, layer_weighting=True, interp_depth = None):
    """
    Compute the extent and intensity profiles of extreme anomalies in the Niño 3.4 region.

    The mean and standard deviation of the whole input region are weighted by cos(latitude) and, with layer_weighting, by
    layer thickness. Anomalies beyond the mean + 2 standard deviations (El Niño) or below the mean - 2 standard deviations
    (La Niña) are kept, optionally interpolated to interp_depth, and summarised within the Niño 3.4 region (5S-5N, 190-240E).

    Parameters
    ----------
    ds (xr.DataArray)         : Composite with dims (depth, lat, lon).
    phase (str)               : 'EL-NINO' or 'LA-NINA'.
    layer_weighting (bool)    : Also weight the threshold statistics by layer thickness. Default is True.
    interp_depth (np.ndarray) : Target depths (m) of the interpolation applied after thresholding. Default is None (none).

    Returns
    -------
    xr.Dataset : 'EXTENT' (meridional extent of the extremes in degrees of latitude, averaged over the longitudes where they
                 occur) and 'INTENSITY' (cos(latitude)-weighted mean anomaly of the extremes), both as profiles over depth.
    """
    # Cosine-of-latitude weights
    lat_weights = np.cos(np.deg2rad(ds['lat']))

    # Vertical layer-thickness weights from the depth midpoints
    z = ds['depth'].values
    bounds = np.concatenate([[0], (z[1:] + z[:-1]) / 2, [z[-1] + (z[-1] - z[-2]) / 2]])
    thickness_weights = xr.DataArray(np.diff(bounds), coords={'depth': ds['depth']}, dims='depth')

    if not layer_weighting:
        weights = lat_weights
    else:
        weights = lat_weights * thickness_weights

    # Weighted mean and standard deviation over the full region
    weighted_ds = ds.weighted(weights)
    mu = weighted_ds.mean().item()
    sigma = weighted_ds.std().item()

    # Phase-specific extreme-anomaly threshold (2 standard deviations)
    if phase == 'EL-NINO':
        thresholded = ds.where(ds > mu + 2 * sigma)
    elif phase == 'LA-NINA':
        thresholded = ds.where(ds < mu - 2 * sigma)
    else:
        raise ValueError("phase must be 'EL-NINO' or 'LA-NINA'")

    # Interpolating to a finer grid, after the thresholding
    if interp_depth is not None:
        thresholded = thresholded.interp(depth=interp_depth).bfill(dim='depth')

    # Restrict the calculation to the Niño 3.4 region
    n34 = thresholded.sel(lat=slice(-5, 5), lon=slice(190, 240))

    # Latitude-weighted anomaly intensity at each depth
    lat_weights_n34 = np.cos(np.deg2rad(n34['lat']))
    anom_intensity = n34.weighted(lat_weights_n34).mean(dim=['lat', 'lon'])

    # Number of anomalous latitude grid cells at each depth and longitude (NaN where there are none)
    grid_count = (~np.isnan(n34)).sum(dim='lat')
    grid_count = grid_count.where(grid_count != 0)

    # Zonally averaged meridional extent
    anom_extent = ((n34['lat'][-1] - n34['lat'][0]) * (grid_count / n34.sizes['lat']).mean(dim='lon'))

    # Extent and intensity as depth-dependent variables
    return xr.merge([anom_extent.rename('EXTENT'), anom_intensity.rename('INTENSITY')])

In [ ]:
# =============================================================================
# TWO-LEVEL HIERARCHICAL BOOTSTRAP OF GROUP-MEAN COMPOSITES AND DIFFERENCES
# =============================================================================
# Helper function to bootstrap a group-mean composite (M = mean_A, B=None) or a group-mean difference (D = mean_A - mean_B)
def hierarchical_bootstrap(A, B, phase, N=N_BOOT, seed=0, paired=False, alpha=0.05, period_labels = (PERIOD_LABELS['HIST'], PERIOD_LABELS['HIST']),
                           return_boot=False, kind='sst', null=0.0, interp_depth = None,**load_kwargs):
    """
    Perform a two-level hierarchical bootstrap of a group-mean composite or of a group-mean difference.

    Level 1 resamples the members (datasets) of each group with replacement, level 2 the events within each drawn member.
    The statistic is the group mean M = mean_A when B is None, otherwise the difference D = mean_A - mean_B. With paired=True,
    B reuses A's member draws. The OBS and NPD groups are first consolidated into one event-paired member (paired_mean), so
    for them only level 2 resamples. The p-value is two-sided against 'null'. The percentile confidence interval is computed
    when B is None or return_boot=True. For kind='3d', the extreme anomalies are masked inside group_mean and boot_mean, with
    thresholds computed from the resampled events.

    Parameters
    ----------
    A, B (list of str)        : Dataset names of group A and of the reference group B (None for a single-group mean).
    phase (str)               : 'EL-NINO' or 'LA-NINA'.
    N (int)                   : Number of bootstrap draws. Default is N_BOOT.
    seed (int)                : Seed of the random number generator. Default is 0.
    paired (bool)             : Reuse A's level-1 draws for B (A and B must list the same models in the same order).
                                Default is False.
    alpha (float)             : Significance level of 'sig' and of the (1 - alpha) interval. Default is 0.05.
    period_labels (tuple)     : Period labels (PERIOD_LABELS values) of the files of A and of B. Default is HIST_EVAL for both.
    return_boot (bool)        : Also return the bootstrap draws as 'boot_var'. Default is False.
    kind (str)                : 'sst', '3d' or 'hov', as in load_data. Default is 'sst'.
    null (float)              : Null value of the p-value. Default is 0.0.
    interp_depth (np.ndarray) : Target depths (m) for '3d' and 'hov'. Default is None.
    **load_kwargs             : Further arguments passed to load_data.

    Returns
    -------
    xr.Dataset or dict : Variables 'M' (B is None) or 'D', 'p' (p-value) and 'sig' (p <= alpha), plus 'lo' and 'hi' (confidence
                         interval) when B is None or return_boot=True, and 'boot_var' when return_boot=True. For kind='3d', a
                         dict of two such Datasets, under 'EXTENT' and 'INTENSITY'.
    """
    groups = [A] if B is None else [A, B]
    if paired and B is None:
        raise ValueError('paired=True requires a reference group B')
    rng = np.random.default_rng(seed)

    # Helper function to load the members of one group as a list of DataArrays (event_id, ...)
    def load_group(names, i):
        """
        Load the members of one group; the EN4 + ORAS5 and the two NPD runs are returned as ONE event-paired member.
        """
        members = {_: load_data(_, phase, kind, interp_depth = interp_depth, period_label = period_labels[i], **load_kwargs).load() for _ in names}
        if set(names) == {'EN4', 'ORAS5'} or set(names) == {'NPD_eORCA025_ERA5', 'NPD_eORCA025_JRA55'}:
            # The grids must match exactly; event_id is outer-joined because the two products number their events differently
            member_A, member_B = xr.align(*members.values(), join='exact', exclude=['event_id', 'depth'])
            both = xr.concat([member_A, member_B], dim=pd.Index(names, name='dataset'), join='outer')
            return [paired_mean(both, phase)]
        # All members must share the non-event grid exactly (raises an error instead of silently inner-joining)
        return list(xr.align(*members.values(), join='exact', exclude=['event_id', 'depth']))

    G_ev = [load_group(g, i) for i, g in enumerate(groups)] # [A_ev] or [A_ev, B_ev]
    if paired and len(G_ev[0]) != len(G_ev[1]):
        raise ValueError('paired=True requires A and B to list the same models in the same order')

    # Helper function to compute the group mean: the mean over members of the member composites (as in the plotted group mean)
    def group_mean(evs):
        """
        Mean over the members of the per-member composites (extreme-anomaly profiles for kind='3d').
        """
        if kind != '3d':
            lst = [e.mean('event_id') for e in evs]
        else:
            lst = [extreme_anomaly_operator(e.mean('event_id'), phase = phase, interp_depth = interp_depth) for e in evs]
        return xr.concat(lst, dim = 'idx').mean(dim = 'idx')

    # Helper function to compute one bootstrap group mean: average of the members drawn at level 1 (idx), each with resampled events
    def boot_mean(evs, idx):
        """
        Mean over the members drawn at level 1 (idx), each with its events resampled at level 2.
        """
        if kind != '3d':
            lst = [resample_events(evs[i]) for i in idx]
        else:
            lst = [extreme_anomaly_operator(resample_events(evs[i]), phase, interp_depth = interp_depth) for i in idx]
        return xr.concat(lst, dim = 'idx').mean(dim = 'idx')

    # Helper function to resample the events of one member (level 2)
    def resample_events(e):
        """
        Mean of n events drawn with replacement from a member (one draw applied to the whole field).
        """
        n = e.sizes['event_id']
        return e.isel(event_id=rng.integers(0, n, size=n)).mean('event_id')

    # Helper function to combine the per-group means into the bootstrapped statistic
    def operation(means):
        """
        Statistic from the per-group means: M = mean_A (one group) or D = mean_A - mean_B.
        """
        return means[0] if len(means) == 1 else means[0] - means[1]

    D = operation([group_mean(evs) for evs in G_ev])
    n_le = xr.zeros_like(D) # valid bootstrap draws <= null
    n_ge = xr.zeros_like(D) # valid bootstrap draws >= null
    n_ok = xr.zeros_like(D) # valid (non-NaN) bootstrap draws
    keep_boot = return_boot or B is None
    if keep_boot:
        boot = []

    for b in tqdm(range(N), leave = False, desc='Bootstrapping: '):
        # Level 1: draw which members enter each group (with replacement); with paired=True, B reuses A's draw
        idx = []
        for evs in G_ev:
            idx.append(idx[0] if paired and idx else rng.integers(0, len(evs), size=len(evs)))
            # The OBS and NPD groups hold one consolidated member (len(evs) = 1), so their draw is always [0] and only level 2 resamples

        # Level 2: resample the events within each drawn member, then average the drawn members
        fake_D = operation([boot_mean(evs, i) for evs, i in zip(G_ev, idx)])
        n_le += (fake_D <= null)
        n_ge += (fake_D >= null)
        n_ok += fake_D.notnull() # NaN compares False, so it must also be left out of the denominator
        if keep_boot:
            boot += [fake_D.expand_dims({'boot':[b]})]

    # Two-sided percentile-bootstrap p-value: twice the smaller tail, +1 so that p is never zero; NaN where D is undefined
    p = (2.0 * (np.minimum(n_le, n_ge) + 1) / (n_ok + 1)).clip(max=1.0).where(D.notnull() & (n_ok > 0))
    out = {'M' if B is None else 'D': D, 'p': p, 'sig': p <= alpha}
    if keep_boot:
        # Percentile confidence interval of the statistic; NaN where D is undefined (land / no data)
        boot = xr.concat(boot, dim='boot', coords = 'minimal', compat = 'override')
        q = boot.quantile([alpha / 2, 1 - alpha / 2], dim='boot')
        out['lo'], out['hi'] = q.isel(quantile=0, drop=True).where(D.notnull()), q.isel(quantile=1, drop=True).where(D.notnull())
        if return_boot:
            out['boot_var'] = boot

    if kind != '3d':
        out = xr.merge([out[_].rename(_) for _ in out.keys()])
    else:
        extent = xr.merge([out[_]['EXTENT'].rename(_) for _ in out.keys()])
        intensity = xr.merge([out[_]['INTENSITY'].rename(_) for _ in out.keys()])
        out = {'EXTENT': extent, 'INTENSITY': intensity}
        del extent, intensity

    return out

In [19]:
# =============================================================================
# BOOTSTRAP OF SSTA COMPOSITES AND NIÑO 3.4 HOVMÖLLERS FOR ALL COMPARISONS
# =============================================================================
# Helper function to bootstrap all group means or all group differences for both ENSO phases
def build_results(kind='sst', bias=False, N=N_BOOT, seed=42, interp_depth = None, return_boot=False):
    """
    Run hierarchical_bootstrap for every comparison and both ENSO phases, and stack the results.

    A comparison is either a group mean (M = mean_A, bias=False) or a group-mean difference (D = mean_A - mean_B, bias=True).
    OBS and NPD are each consolidated into one event-paired member. SCEN - HIST and HIST - CTRL are paired by model,
    the other differences are unpaired. All comparisons and both phases use the same seed.

    Parameters
    ----------
    kind (str)                : 'sst' (surface SSTA composites) or 'hov' (Niño 3.4 Hovmöllers), as in load_data. Default is 'sst'.
    bias (bool)               : True for the six differences (HIST-OBS, CTRL-OBS, NPD-OBS, SCEN_F-HIST, SCEN_T-HIST, HIST-CTRL),
                                False for the six group means (CTRL, HIST, SCEN_F, SCEN_T, NPD, OBS). Default is False.
    N (int)                   : Number of bootstrap draws per comparison. Default is N_BOOT.
    seed (int)                : Seed of the random number generator of each bootstrap. Default is 42.
    interp_depth (np.ndarray) : Target depths (m) for 'hov'. Default is None.
    return_boot (bool)        : Also keep the bootstrap draws ('boot_var'). Default is False.

    Returns
    -------
    xr.Dataset : Output of hierarchical_bootstrap with new dims 'phase' and 'comp' (the comparison names above).
    """
    comparisons = (
        [('HIST-OBS', HIGHRESMIP_EXPS_HIST, OBS, {}),
         ('CTRL-OBS', HIGHRESMIP_EXPS_CTRL, OBS, {}),
         ('NPD-OBS', NPD_EXPS, OBS, {}),
         ('SCEN_F-HIST', HIGHRESMIP_EXPS_SCEN, HIGHRESMIP_EXPS_HIST, {'period_labels': (PERIOD_LABELS['SCEN_F'], PERIOD_LABELS['HIST']), 'paired': True}),
         ('SCEN_T-HIST', HIGHRESMIP_EXPS_SCEN, HIGHRESMIP_EXPS_HIST, {'period_labels': (PERIOD_LABELS['SCEN_T'], PERIOD_LABELS['HIST']), 'paired': True}),
         ('HIST-CTRL', HIGHRESMIP_EXPS_HIST, HIGHRESMIP_EXPS_CTRL, {'paired': True})]

         if bias else
         [('CTRL', HIGHRESMIP_EXPS_CTRL, None, {}),
          ('HIST', HIGHRESMIP_EXPS_HIST, None, {}),
          ('SCEN_F', HIGHRESMIP_EXPS_SCEN, None, {'period_labels': (PERIOD_LABELS['SCEN_F'], None)}),
          ('SCEN_T', HIGHRESMIP_EXPS_SCEN, None, {'period_labels': (PERIOD_LABELS['SCEN_T'], None)}),
          ('NPD', NPD_EXPS, None, {}),
          ('OBS', OBS, None, {})])

    phase_results = []

    for phase in tqdm(('EL-NINO', 'LA-NINA'), leave=False, desc = 'Phase: '):
        comp_results = []
        for name, A, B, options in tqdm(comparisons, leave=False, desc = 'Comp: '):
            result = hierarchical_bootstrap(A=A, B=B, phase=phase, N=N, seed=seed, kind=kind, interp_depth = interp_depth,
                                            return_boot = return_boot, **options,)
            comp_results.append(result.expand_dims(comp=[name]))

        phase_results.append(xr.concat(comp_results, dim='comp',
                                       coords = 'minimal', compat = 'override').expand_dims(phase=[phase]))

    results = xr.concat(phase_results, dim='phase')
    return results

In [21]:
# Bootstrap of the Niño 3.4 Hovmöllers (top 300 m, 0.5 m depth grid) and of the surface SSTA composites: first the six differences
# (bias=True), then the six group means (bias=False). Each run has its own seed (10, 20, 30, 40)
res_bias_hov = build_results(kind='hov', bias=True, seed=10, interp_depth = np.arange(0, 300.1, 0.5), return_boot=False)
print('Finished bootstrapping Hovmoeller composites (bias=True) for all comparisons.')
res_hov = build_results(kind='hov', bias=False, seed=20, interp_depth = np.arange(0, 300.1, 0.5), return_boot=False)
print('Finished bootstrapping Hovmoeller composites (bias=False) for all comparisons.')
res_bias_ssta = build_results(kind='sst', bias=True, seed=30, interp_depth = None, return_boot=False)
print('Finished bootstrapping SSTA composites (bias=True) for all comparisons.')
res_ssta = build_results(kind='sst', bias=False, seed=40, interp_depth = None, return_boot=False)
print('Finished bootstrapping SSTA composites (bias=False) for all comparisons.')

Phase:   0%|          | 0/2 [00:00<?, ?it/s]

Comp:   0%|          | 0/6 [00:00<?, ?it/s]

Bootstrapping:   0%|          | 0/2000 [00:00<?, ?it/s]

Bootstrapping:   0%|          | 0/2000 [00:00<?, ?it/s]

Bootstrapping:   0%|          | 0/2000 [00:00<?, ?it/s]

Bootstrapping:   0%|          | 0/2000 [00:00<?, ?it/s]

... (50 outputs left out)

/dssgfs01/working/sg13n23/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Bootstrapping:   0%|          | 0/2000 [00:00<?, ?it/s]

/dssgfs01/working/sg13n23/miniforge3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Finished bootstrapping SSTA composites (bias=False) for all comparisons.


In [22]:
# Saving the processed data (read by appendix_B.ipynb and plotting.ipynb). Along 'comp', the six group means (CTRL, HIST, SCEN_F, SCEN_T,
# NPD, OBS) are followed by the six differences (HIST-OBS, CTRL-OBS, NPD-OBS, SCEN_F-HIST, SCEN_T-HIST, HIST-CTRL); 'M' is NaN for
# the differences and 'D' for the group means. Dims: (phase, comp, lat, lon) for the SSTA file, with 30S-30N and 120-290E, and
# (phase, comp, time, depth) for the Hovmöller file, with the lag time (-14 to +8 months) and 0-300 m
xr.concat([res_ssta, res_bias_ssta], dim = 'comp').to_netcdf(f'{DATA_DIR}ENSO_SSTA_COMP_DIFF_RESULTS_WITH_SIG.nc')
xr.concat([res_hov, res_bias_hov], dim = 'comp').to_netcdf(f'{DATA_DIR}ENSO_N34_HOV_COMP_DIFF_RESULTS_WITH_SIG.nc')

## Appendix A.2: Surface variance ratios and subsurface metrics

### Testing variance ratios using a block permutation test

In [ ]:
# Niño 3.4 indices: HIST_EVAL (1979-2014, 432 months; HIST and CTRL runs), FUTR_EVAL (2015-2050, 432 months; SCEN) and TRNC_SCEN (2015-2040, 312 months; SCEN)
df_HIST = pd.read_csv(f'{DATA_DIR}N34_INDEX_HIST_EVAL_1979-2014.csv', index_col = 0, parse_dates = True)[HIGHRESMIP_EXPS_HIST + HIGHRESMIP_EXPS_CTRL]
df_SCEN_F = pd.read_csv(f'{DATA_DIR}N34_INDEX_FUTR_EVAL_2015-2050.csv', index_col = 0, parse_dates = True)[HIGHRESMIP_EXPS_SCEN]
df_SCEN_T = pd.read_csv(f'{DATA_DIR}N34_INDEX_TRNC_SCEN_2015-2040.csv', index_col = 0, parse_dates = True)[HIGHRESMIP_EXPS_SCEN]

In [ ]:
# =============================================================================
# BLOCK PERMUTATION TEST OF THE NIÑO 3.4 VARIANCE RATIOS BETWEEN EXPERIMENTS
# =============================================================================
# For each model, the 24-month blocks of the numerator and denominator series are pooled and randomly reassigned to the two series
# (resampling without replacement) N_BOOT times. This gives the null distribution of the variance ratio, against which the actual
# ratio is tested (two-sided)
BLOCK_SIZE = 24 # Block length in months (2 years)

# Comparisons: (numerator DataFrame, column suffix, denominator DataFrame, column suffix)
COMPARISONS = {'HIST/CTRL': (df_HIST, '_HIST', df_HIST, '_CTRL'),
               'SCEN_F/HIST': (df_SCEN_F, '_SCEN', df_HIST, '_HIST'),
               'SCEN_T/HIST': (df_SCEN_T, '_SCEN', df_HIST, '_HIST')}

# One random number generator per comparison, with fixed seeds (chosen arbitrarily) for reproducibility
rngs = dict(zip(COMPARISONS, [np.random.default_rng(s) for s in [24, 42, 67]]))

# Variance ratios of the null distribution, actual variance ratios and p-values, by comparison and model
boot_var_ratio = {comp: {} for comp in COMPARISONS}
actual_var_ratio = {comp: {} for comp in COMPARISONS}
p_value = {comp: {} for comp in COMPARISONS}

for comp, (df_NUM, sfx_NUM, df_DEN, sfx_DEN) in COMPARISONS.items():
    rng = rngs[comp]
    for model in tqdm(HIGHRESMIP_MODELS, desc = f'{comp} models: '):
        series_NUM = df_NUM[model + sfx_NUM].values
        series_DEN = df_DEN[model + sfx_DEN].values
        N_BLOCKS_NUM = len(series_NUM) // BLOCK_SIZE
        N_BLOCKS_DEN = len(series_DEN) // BLOCK_SIZE
        # Actual variance ratio (sample variances)
        actual_var_ratio[comp][model] = np.var(series_NUM, ddof = 1)/np.var(series_DEN, ddof = 1)
        # Pool of the blocks of both series: (N_BLOCKS_NUM + N_BLOCKS_DEN) rows of BLOCK_SIZE months
        block_pool = np.concatenate([series_NUM.reshape(N_BLOCKS_NUM, -1), series_DEN.reshape(N_BLOCKS_DEN, -1)])
        # One random permutation of the pooled blocks per row: (N_BOOT, N_BLOCKS_NUM + N_BLOCKS_DEN)
        idx = rng.permuted(np.tile(np.arange(block_pool.shape[0]), (N_BOOT, 1)), axis = 1)
        # The first N_BLOCKS_NUM blocks form the fake numerator series, the rest the fake denominator series
        fake_NUM = block_pool[idx[:, :N_BLOCKS_NUM]].reshape(N_BOOT, -1)
        fake_DEN = block_pool[idx[:, N_BLOCKS_NUM:]].reshape(N_BOOT, -1)
        boot = np.var(fake_NUM, axis = 1, ddof = 1)/np.var(fake_DEN, axis = 1, ddof = 1)
        boot_var_ratio[comp][model] = boot # Kept for inspection; not used below
        # Two-sided p-value: twice the smaller tail, +1 so that p is never zero
        p_hi = (1 + np.sum(boot >= actual_var_ratio[comp][model])) / (N_BOOT + 1)
        p_lo = (1 + np.sum(boot <= actual_var_ratio[comp][model])) / (N_BOOT + 1)
        p_value[comp][model] = min(1.0, 2 * min(p_hi, p_lo))

HIST/CTRL models:   0%|          | 0/6 [00:00<?, ?it/s]

SCEN_F/HIST models:   0%|          | 0/6 [00:00<?, ?it/s]

SCEN_T/HIST models:   0%|          | 0/6 [00:00<?, ?it/s]

In [25]:
# Models with a significant variance ratio (p <= ALPHA) for each comparison; displayed only, not saved
ns_lst_var_rat = {}
for comp in COMPARISONS.keys():
    ns_lst_var_rat[comp] = [model for model, pval in p_value[comp].items() if pval <= ALPHA]

print('Models with significant variance ratios (p <= 0.05) for each comparison:\n',ns_lst_var_rat)

Models with significant variance ratios (p <= 0.05) for each comparison:
 {'HIST/CTRL': ['EC-Earth3P-HR'], 'SCEN_F/HIST': ['HadGEM3-GC31-HH', 'CNRM-CM6-1-HR'], 'SCEN_T/HIST': ['HadGEM3-GC31-HH', 'CNRM-CM6-1-HR']}


### Bootstrapping for extreme temperature anomaly extent and intensity profiles for all group-mean and group-mean difference comparisons

**Note**: The extreme temperature anomaly bootstrap was run as a separate script with a corresponding SLURM script, with one array task per phase-group combination, because each combination took too long to run in sequence. The cell below is a copy of that script, shown for reference only: it reads its task index from `sys.argv`, so there is no point in running it in the notebook. The script itself (`appendix_A_ext_temp_prof_bootstrapping.py`) is not part of this repository; it consisted of this cell plus the imports, constants and functions defined above. Its output files are read by appendix_B.ipynb and plotting.ipynb.

In [ ]:
# =============================================================================
# BOOTSTRAP OF EXTREME TEMPERATURE ANOMALY EXTENT AND INTENSITY PROFILES (ONE SLURM TASK)
# =============================================================================
# Datasets of each group label (SCEN_F and SCEN_T use the same SCEN runs, over different periods)
lst_dict = {'CTRL': HIGHRESMIP_EXPS_CTRL, 'HIST': HIGHRESMIP_EXPS_HIST, 'SCEN_F': HIGHRESMIP_EXPS_SCEN, 'SCEN_T': HIGHRESMIP_EXPS_SCEN, 'NPD': NPD_EXPS, 'OBS': OBS}

# Task index of this job (SLURM_ARRAY_TASK_ID in array runs)
idx = int(sys.argv[1])
# Job list: 12 group-mean jobs (6 labels x 2 phases) followed by 10 difference jobs (5 non-OBS labels x 2 phases) = 22 jobs (0-21)
jobs = ([(label, phase, False) for label in lst_dict for phase in ['EL-NINO', 'LA-NINA']] +
        [(label, phase, True)  for label in lst_dict if label != 'OBS' for phase in ['EL-NINO', 'LA-NINA']])
N = N_BOOT # Number of bootstrap draws

# Parameters of this job: group label, ENSO phase and whether to compute the difference to the reference group
label, phase, bias_calc = jobs[idx]

# Seed of this job (in the file name as s<seed>): one base per label (shared by SCEN_F and SCEN_T), +10 for differences, +1 for La Niña
bases = {'CTRL': 0, 'HIST': 2, 'SCEN_F': 4, 'SCEN_T': 4, 'NPD': 6, 'OBS': 8}
seed = bases[label] + (10 if bias_calc else 0) + (0 if phase == 'EL-NINO' else 1)

p1 = PERIOD_LABELS.get(label, PERIOD_LABELS['HIST']) # Period of group A: the scenario labels have their own, all others use HIST_EVAL
is_scen = label in ('SCEN_F', 'SCEN_T')

# Period of the reference group (HIST_EVAL for differences), pairing by model, and reference group B (None for group means):
# SCEN is compared with HIST (paired by model), all other groups with OBS
period_labels = (p1, PERIOD_LABELS['HIST'] if bias_calc else None)
paired = is_scen if bias_calc else None
lst_b = (HIGHRESMIP_EXPS_HIST.copy() if is_scen else OBS.copy()) if bias_calc else None

# Log the job configuration before running the bootstrap
print(f"Running with label={label}, phase={phase}, bias_calc={bias_calc} | N = {N}, seed = {seed} | period_labels = {period_labels} | paired = {paired}")

# Run the 3D hierarchical bootstrap on the 0-300 m depth grid (10 m spacing)
out = hierarchical_bootstrap(A=lst_dict[label], B=lst_b, phase=phase, N=N, seed=seed, return_boot=True,
                             kind='3d', interp_depth=np.arange(0, 300.1, 10), paired=paired, period_labels=period_labels)
# Stack the INTENSITY and EXTENT outputs along a new 'type' dimension
out = xr.concat([out['INTENSITY'].expand_dims(type=['INTENSITY']), out['EXTENT'].expand_dims(type=['EXTENT'])], dim='type')
# 'BIAS_' in the file name marks differences rather than group means
tag = 'BIAS_' if bias_calc else ''
# Write the combined bootstrap output: dims (type, depth, boot) with variables D or M, p, sig, lo, hi and boot_var
out.to_netcdf(f'{DATA_DIR}EXT_TEMP_BOOTSTRAP/ENSO_EXT_TEMP_{tag}{label}_{phase}_b{N:04d}s{seed:02d}.nc')

print('Done!')

'''
SLURM SCRIPT TO RUN THE ABOVE SNIPPET + DEPENDENT FUNCTIONS/VARS, 
IN PARALLEL (ARRAY JOB) MODE ON A HPC CLUSTER
-----------------------------------------------------------------
#!/bin/bash
#SBATCH --partition=compute
#SBATCH --job-name=anom_boot
#SBATCH -o %A_%a.out
#SBATCH -e %A_%a.err
#SBATCH --time=11:55:00
#SBATCH --mem=128000
#SBATCH --array=0-21

# Add any scripts you want to run here, e.g.:
python3 appendix_A_ext_temp_prof_bootstrapping.py $SLURM_ARRAY_TASK_ID

exit
'''

## Appendix A.3: Power spectral analysis

In [ ]:
# Every HighResMIP experiment: each model with each experiment type (the model, NPD and OBS lists are defined above)
HIGHRESMIP_EXPS = [f'{model}_{exp_type}' for model, exp_type in product(HIGHRESMIP_MODELS, ['CTRL', 'HIST', 'SCEN'])]

# Welch settings used in the paper: 14-yr (168-month) Hann segments, 75% overlap, zero-padded to 3 times the segment length
PSD_KW = dict(nperseg=168, period_min=0.5, period_max=14.0, fs=12.0, noverlap_frac=0.75, nfft=3*168)

# Band searched for the dominant period (0.5-14 yr). The red-noise test and the shift test both use it,
# so the period that is tested is the same period reported in Tables 2-3 and Figs. 2i/7g
PSD_BAND = (PSD_KW['period_min'], PSD_KW['period_max'])

# Spacing of the zero-padded Welch frequency grid: fs / nfft = 12 / 504 = 1/42 cycles per year.
# Every dominant frequency is a whole number k of these steps, and so is every shift between two runs
DF = PSD_KW['fs'] / PSD_KW['nfft']

N_SIMS = 2000 # Number of red-noise series / phase-randomised surrogates per test (i.e., size of each null distribution)

### AR(1) red noise
$x_t = \varphi \cdot x_{t-1} + \varepsilon_t$, where
- $x_t$, $x_{t-1}$: values of the series at timesteps $t$ and $t-1$
- $\varphi$: memory coefficient, the fraction of last step's value that carries over. For AR(1) it <br> equals the lag-1 
autocorrelation, so it is estimated from the lag-1 autocorrelation of the (linearly detrended) data ($|\varphi| < 1$).
- $\varepsilon_t$: a random "kick" (white noise): zero mean, independent of everything before it, <br> randomly sampled 
from a Gaussian distribution with $\sigma_\varepsilon = \sigma_x\sqrt{1-\varphi^2}$.

To generate one red-noise series (one realisation) that mimics an input time series, the following are needed:
- **Lag-1 autocorrelation ($\varphi$):** Measure of how related each value in the input timeseries is to the one before it (i.e., how related $x_{t-1}$ and $x_{t}$ are).
- **Standard deviation of the input ($\sigma_{x}$):** Used to draw the first value from a zero-mean Gaussian with this spread, so the red noise series starts already from a plausible random point i.e., from the zero-centred Gaussian distribution that shares its variance with the input time-series data
- **Random "kick" size, $\sigma_\varepsilon = \sigma_x\sqrt{1-\varphi^2}$:** smaller than $\sigma_{x}$. Each kick fades only slowly, so at any moment the series holds the sum of many recent kicks, making it more variable than a single kick. Shrinking the kicks by $\sqrt{1-\varphi^2}$ makes the *output (red-noise) series* match the input's variance. Too big of a $\sigma_\varepsilon$ can create fake series too variable → test too strict; too small → too lenient.

In [27]:
# Helper function to create one AR(1) red-noise series that mimics the Niño 3.4 SST anomaly time series
def red_noise(phi, sigma_n34, n, rng):
    """
    Create one AR(1) red-noise series of length n, similar to the Niño 3.4 SST anomaly time series.

    Parameters
    ----------
    phi (float)               : Lag-1 autocorrelation (memory coefficient) of the series to mimic.
    sigma_n34 (float)         : Standard deviation of the series to mimic.
    n (int)                   : Length of the series.
    rng (np.random.Generator) : Random number generator, shared between series so that each one gets new noise.

    Returns
    -------
    np.ndarray : Red-noise series x_t = phi * x_(t-1) + eps_t, with the variance of the input series in expectation.
    """
    red = np.zeros(n)
    red[0] = rng.normal(0.0, sigma_n34) # Initial condition x_0, from a zero-mean Gaussian with the standard deviation of the input

    # The noise term eps_t has a smaller standard deviation than the input: each kick fades only slowly (by a factor phi per
    # step), so the series holds the sum of many recent kicks. Shrinking them by sqrt(1 - phi^2) gives the series the input's variance
    sigma_eps = sigma_n34 * np.sqrt(1 - phi**2)
    eps = rng.normal(0.0, sigma_eps, n) # One kick per timestep (eps[0] is not used)
    for t in range(1, n):
        red[t] = phi * red[t-1] + eps[t] # Keep a fraction phi of the last month, add a new kick
    return red


# Helper function to test the Welch PSD of a series against AR(1) red noise with the same memory and spread
def red_noise_mc_test(series, psd_kw, n_sims=1000, band=PSD_BAND, seed=0):
    """
    Test the Welch PSD of a series against n_sims AR(1) red-noise series with the same memory and spread.

    The dominant peak of the series within the band is compared with the dominant peaks of the red-noise series, each
    expressed relative to the median red-noise PSD at its own frequency.

    Parameters
    ----------
    series (pd.Series) : Time series (e.g. Niño 3.4 SST anomalies); NaNs are dropped.
    psd_kw (dict)      : Welch settings passed to compute_psd (see PSD_KW).
    n_sims (int)       : Number of red-noise series. Default is 1000 (the notebook passes N_SIMS).
    band (tuple)       : Period band (years) searched for the dominant period. Default is PSD_BAND.
    seed (int)         : Seed of the random number generator. Default is 0.

    Returns
    -------
    dict : 'period' and 'Pxx' (Welch PSD of the series), 'p05', 'median' and 'p95' (percentiles of the red-noise PSDs),
           'phi' and 'sigma' (lag-1 autocorrelation and standard deviation copied by the red noise), 'dominant_period' (yr),
           'p_peak' (fraction of red-noise series with a dominant peak at least as high) and 'sim_peak' (the red-noise
           peak heights, relative to the median).
    """
    x = series.dropna()

    # The AR(1) parameters are estimated from the linearly detrended series: a trend left in the anomalies (e.g. residual
    # warming at the end of SCEN) would distort the lag-1 autocorrelation and the variance that the red noise copies.
    # The spectrum is still computed from x: Welch removes a linear trend from every 14-yr segment, and a global linear
    # trend is linear within each segment, so detrending x first would give the same PSD
    x_dt = pd.Series(detrend(x.values.astype(float), type='linear'), index=x.index)
    phi, sigma = x_dt.autocorr(lag=1), x_dt.std() # Memory and spread to copy

    rng = np.random.default_rng(seed = seed) # Created once, so that every red-noise series gets new kicks
    # Welch PSD of the real series (also defines the period grid)
    period, Pxx, _, _ = compute_psd(x, **psd_kw)

    # Welch PSD of each red-noise series, with exactly the same settings as the real series
    sims = np.array([compute_psd(pd.Series(red_noise(phi, sigma, len(x), rng)), **psd_kw)[1] for _ in tqdm(range(n_sims), desc='Red-noise simulations', leave = False)])

    # Red-noise envelope: the median is the background, p05 and p95 the pointwise 5-95% range (visual guide only)
    p05, median, p95 = np.percentile(sims, [5, 50, 95], axis=0)

    in_band = (period >= band[0]) & (period <= band[1]) # Band searched for the dominant period (PSD_BAND)

    # Peak test: is the dominant peak higher, relative to red noise, than the dominant peaks of red noise itself?
    k = Pxx[in_band].argmax() # Bin of the dominant period (highest PSD in the band)
    dominant_period = period[in_band][k]
    peak_score = Pxx[in_band][k] / median[in_band][k] # Height of the dominant peak relative to the red-noise background
    k_sim = sims[:, in_band].argmax(axis=1) # Dominant bin of each red-noise series in the band
    sim_peak = sims[:, in_band][np.arange(n_sims), k_sim] / median[in_band][k_sim] # ... and its height relative to the background
    p_peak = (1 + np.sum(sim_peak >= peak_score)) / (1 + n_sims) # Fraction of red-noise series with a dominant peak at least as high

    return dict(period=period, Pxx=Pxx, p05=p05, median=median, p95=p95, phi=phi, sigma=sigma,
                p_peak=p_peak, sim_peak=sim_peak, dominant_period=dominant_period)

In [28]:
# =============================================================================
# LOADING THE NIÑO 3.4 INDICES (HIST_EVAL AND FUTR_EVAL PERIODS)
# =============================================================================
df_hist_eval = pd.read_csv(f'{DATA_DIR}N34_INDEX_HIST_EVAL_1979-2014.csv', index_col=0, parse_dates=True)
df_futr_eval = pd.read_csv(f'{DATA_DIR}N34_INDEX_FUTR_EVAL_2015-2050.csv', index_col=0, parse_dates=True)

# Stack the two periods along time: the HIST_EVAL columns (CTRL, HIST, NPD, OBS) are NaN in 2015-2050 and the SCEN columns
# are NaN in 1979-2014 (the NaNs are dropped in red_noise_mc_test)
df = pd.concat([df_hist_eval, df_futr_eval], axis=0)

In [29]:
# =============================================================================
# RED-NOISE TEST FOR EVERY MODEL EXPERIMENT, NPD RUN AND OBS DATASET
# =============================================================================
res = {}
# Datasets whose dominant PSD peak is not significant against red noise (p > ALPHA); they are highlighted in the figures
ns_lst = []
for col in tqdm(HIGHRESMIP_EXPS + NPD_EXPS + OBS):
    res[col] = red_noise_mc_test(df[col], PSD_KW, band=PSD_BAND, n_sims=N_SIMS, seed = 42) # Same fixed seed for every dataset, for reproducibility
    if res[col]['p_peak'] > ALPHA:
        ns_lst.append(col)

  0%|          | 0/22 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

... (13 outputs left out)

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

In [30]:
# Datasets with a non-significant dominant PSD peak (plotting.ipynb keeps a hand-copied version of this list as ns_lst)
print('Non-significant PSD peak list:', ns_lst)

Non-significant PSD peak list: ['CNRM-CM6-1-HR_CTRL', 'CNRM-CM6-1-HR_SCEN', 'MPI-ESM1-2-HR_HIST', 'MPI-ESM1-2-XR_CTRL', 'MPI-ESM1-2-XR_HIST', 'MPI-ESM1-2-XR_SCEN']


In [34]:
# Saving the red-noise test results (read by appendix_C.ipynb): one entry per dataset (22 keys; 'OBS_MEAN' is added to res further
# below, after this save)
with open(f"{DATA_DIR}PSD_PEAK_TEST_RED_NOISE.pkl", "wb") as f:
    pickle.dump(res, f)

### Test of the shift in dominant frequency between two runs (CTRL-HIST, HIST-SCEN, CTRL-OBS, HIST-OBS)

The question this test answers is: if ENSO in the two runs were the same, how often would two 36-year windows give dominant frequencies at least this far apart?

1. Measure the real shift: the absolute difference between the two runs' dominant frequencies. All dominant frequencies lie on the Welch grid (steps of DF = 1/42 cycles/yr), so the shift is counted as a whole number of grid steps.
2. Join the two runs, each linearly detrended on its own, into one 72-year series. The pairs are CTRL then HIST, HIST then SCEN, CTRL then OBS and HIST then OBS, where OBS is the mean of EN4 and ORAS5.
3. Make a fake 72-year series. The Fourier transform splits the series into cycles of every period. The code keeps the size of each cycle and gives it a random timing (phase). The fake series has the same spectrum but different wiggles, and its two halves differ only by chance: this is the "no change" world.
4. Split it at the original junction (first 432 months, rest), find each part's dominant frequency with the same Welch settings and record the absolute difference.
5. Repeat N_SIMS (2000) times. This gives the range of shifts that chance alone produces.
6. p is the fraction of fake shifts at least as large as the real one, (1 + count) / (1 + N_SIMS). The smallest shift with p ≤ 0.05 is the test's critical shift, drawn as the dashed line in Figure C2 (appendix_C.ipynb).

The test is run for every model and each of the four pairs. Where at least one of the two runs has no significant dominant peak (red-noise test), the shift is reported but not interpreted, and the model is marked with '*' in Figure C2.

In [ ]:
# =============================================================================
# PHASE-RANDOMISED SHIFT TEST: HELPER FUNCTIONS
# =============================================================================
# Helper function to find the dominant frequency of a series as a whole number of grid steps
def peak_bin(x):
    """
    Find the dominant frequency of a series as a whole number of steps of the Welch frequency grid (DF).
    """
    # compute_psd returns the dominant PERIOD; 1/period is the dominant frequency in cycles/yr, which always lies on the
    # Welch grid k * DF (DF = 1/42 cycles/yr). Rounding to the integer k removes float noise (~1e-15), so equal shifts
    # compare as exactly equal in the p-value and the critical shift
    _, _, psd_peak_t, _ = compute_psd(pd.Series(x), **PSD_KW) # Dominant period (yr), searched within PSD_BAND
    return int(np.rint((1.0 / psd_peak_t) / DF))

# Helper function to create one phase-randomised surrogate of a single time series
def phase_surrogate(x, rng):
    """
    Create one phase-randomised surrogate of a time series, with the same amplitude spectrum (and so variance) as the input.
    """
    X = np.fft.rfft(x) # Fourier transform: one complex coefficient (amplitude + phase) per frequency
    ph = rng.uniform(0, 2 * np.pi, X.size) # A random phase (timing) for every frequency
    ph[0] = 0.0 # The zero-frequency (mean) term must stay real
    if x.size % 2 == 0:
        ph[-1] = 0.0 # For even lengths, the Nyquist term must also stay real
    return np.fft.irfft(np.abs(X) * np.exp(1j * ph), n=x.size) # Original amplitudes with random phases, back in the time domain

# Helper function to test the shift in dominant frequency from series a to series b against a 'no change' null
def shift_test(a, b, n=5000, seed=0, alpha=ALPHA):
    """
    Test the shift in dominant frequency from series a to series b against a phase-randomised 'no change' null.

    Both series are detrended and joined; phase-randomised surrogates of the joined record are split at the same point and
    the shift between the dominant frequencies of the two parts forms the null distribution.

    Parameters
    ----------
    a, b (array-like) : Series of the two runs (CTRL -> HIST, HIST -> SCEN, CTRL -> OBS or HIST -> OBS), of equal length.
    n (int)           : Number of surrogates. Default is 5000 (the notebook passes N_SIMS).
    seed (int)        : Seed of the random number generator. Default is 0.
    alpha (float)     : Significance level of the critical shift. Default is ALPHA.

    Returns
    -------
    dict : 'shift_bins' and 'shift_cpy' (observed shift in grid steps and in cycles/yr), 'p_value', 'crit_bins' and 'crit_cpy'
           (critical shift, the smallest with p <= alpha), 'null_bins' and 'null_dist' (null distribution in grid steps and in
           cycles/yr), and 'bin_a' and 'bin_b' (dominant frequency of each series in grid steps).
    """
    rng = np.random.default_rng(seed)

    a = detrend(np.asarray(a, dtype=float)) # detrend each series on its own,
    b = detrend(np.asarray(b, dtype=float)) # so that joining them leaves no step
    k_a, k_b = peak_bin(a), peak_bin(b) # dominant frequencies, in grid steps
    obs = abs(k_b - k_a) # observed shift, in grid steps (integer)

    pooled = np.concatenate([a, b]) # one record holding both series
    h = a.size
    null = np.empty(n, dtype=int)
    for i in range(n):
        s = phase_surrogate(pooled, rng) # 'no change' version of the joined record
        null[i] = abs(peak_bin(s[h:]) - peak_bin(s[:h])) # split at the same point

    # Helper function to compute the p-value of a shift of k grid steps
    def p_of(k):
        """
        p-value of an observed shift of k steps: the fraction of 'no change' shifts at least as large.
        """
        return (1 + np.sum(null >= k)) / (1 + n)

    p = p_of(obs)

    # Critical shift: the smallest shift (in steps) whose p-value is <= alpha. It is computed directly from the discrete null,
    # because np.percentile(null, 95) can return a value whose own p-value is above alpha (ties at that value) or a value
    # between grid steps (interpolation). One step past the largest null value always qualifies, as its p-value is 1/(n+1)
    assert 1 / (1 + n) <= alpha, 'n is too small for this alpha'
    crit = next(k for k in range(null.max() + 2) if p_of(k) <= alpha)

    return dict(shift_bins=obs, shift_cpy=obs * DF, # observed shift (steps, cycles/yr)
                p_value=p,
                crit_bins=crit, crit_cpy=crit * DF, # critical shift (steps, cycles/yr)
                null_bins=null, null_dist=null * DF, # null distribution (steps, cycles/yr)
                bin_a=k_a, bin_b=k_b) # dominant frequencies of the two runs (steps)

In [36]:
# =============================================================================
# OBS REFERENCE: MEAN OF EN4 AND ORAS5 (NEEDS ITS OWN RED-NOISE RESULT FOR THE ASSERT AND '*')
# =============================================================================
OBS_KEY  = 'OBS_MEAN' # 'OBS' is already the list ['EN4', 'ORAS5']
obs_mean = df_hist_eval[OBS].mean(axis=1)
res[OBS_KEY] = red_noise_mc_test(obs_mean, PSD_KW, band=PSD_BAND, n_sims=N_SIMS, seed=42)

# =============================================================================
# SHIFT TEST FOR EACH MODEL (CTRL->HIST, HIST->SCEN, CTRL->OBS, HIST->OBS)
# =============================================================================
shift_test_dict = {}
for model in tqdm(HIGHRESMIP_MODELS):
    pairs = [('CTRL-HIST', df_hist_eval[model + '_CTRL'], df_hist_eval[model + '_HIST']),
             ('HIST-SCEN', df_hist_eval[model + '_HIST'], df_futr_eval[model + '_SCEN']),
             ('CTRL-OBS',  df_hist_eval[model + '_CTRL'], obs_mean),
             ('HIST-OBS',  df_hist_eval[model + '_HIST'], obs_mean),]
    for label, a, b in tqdm(pairs, leave = False, desc=f'Processing {model}'):
        out = shift_test(a, b, n=N_SIMS, seed=42)
        exp_a, exp_b = label.split('-')
        # Keys into 'res' for both series ('OBS' maps to the EN4/ORAS5 mean)
        key_a, key_b = [OBS_KEY if exp == 'OBS' else f'{model}_{exp}' for exp in (exp_a, exp_b)]

        # Sanity check: both tests must locate the same dominant frequency in each series (same series, same band)
        for key, k in ((key_a, out['bin_a']), (key_b, out['bin_b'])):
            assert k == int(np.rint((1 / res[key]['dominant_period']) / DF)), f'{key}: dominant frequency differs between tests'

        # A shift is interpretable only if BOTH series have a significant dominant peak in the red-noise test.
        # The test is still run and reported for every pair; pairs failing this are marked with '*' in FIGURE C2
        out['both_peaks_sig'] = all(res[key]['p_peak'] <= ALPHA for key in (key_a, key_b))
        shift_test_dict[(model, label)] = out

Red-noise simulations:   0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

Processing HadGEM3-GC31-HH:   0%|          | 0/4 [00:00<?, ?it/s]

Processing EC-Earth3P-HR:   0%|          | 0/4 [00:00<?, ?it/s]

Processing CNRM-CM6-1-HR:   0%|          | 0/4 [00:00<?, ?it/s]

Processing CMCC-CM2-VHR4:   0%|          | 0/4 [00:00<?, ?it/s]

Processing MPI-ESM1-2-HR:   0%|          | 0/4 [00:00<?, ?it/s]

Processing MPI-ESM1-2-XR:   0%|          | 0/4 [00:00<?, ?it/s]

In [40]:
# Saving the shift-test results (read by appendix_C.ipynb): one entry per (model, pair), 24 in total
with open(f"{DATA_DIR}PSD_PEAK_SHIFT_TEST_PHASE_RANDOM.pkl", "wb") as f:
    pickle.dump(shift_test_dict, f)